In [85]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('sample.pdf')

text = loader.load()

print(len(text))
print(text[0].metadata)
print(text[0].page_content)

23
{'producer': 'Skia/PDF m131 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Deep Learning Curriculum', 'source': 'sample.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1'}
CampusX
Deep
Learning
Curriculum
A.
Artificial
Neural
Network
and
how
to
improve
them
1.
Biological
Inspiration
●
Understanding
the
neuron
structure
●
Synapses
and
signal
transmission
●
How
biological
concepts
translate
to
artificial
neurons
2.
History
of
Neural
Networks
●
Early
models
(Perceptron)
●
Backpropagation
and
MLPs
●
The
"AI
Winter"
and
resurgence
of
neural
networks
●
Emergence
of
deep
learning
3.
Perceptron
and
Multilayer
Perceptrons
(MLP)
●
Single-layer
perceptron
limitations
●
XOR
problem
and
the
need
for
hidden
layers
●
MLP
architecture
4.
Layers
and
Their
Functions
●
Input
Layer
○
Accepting
input
data
●
Hidden
Layers
○
Feature
extraction
●
Output
Layer
○
Producing
final
predictions
5.
Activation
Functions


In [86]:
for page in text:
    page.page_content = " ".join(page.page_content.split())

In [87]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

docs = splitter.split_documents(text)

print(len(docs))
print(docs[0].metadata)
print(docs[0].page_content)

23
{'producer': 'Skia/PDF m131 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Deep Learning Curriculum', 'source': 'sample.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1'}
CampusX Deep Learning Curriculum A. Artificial Neural Network and how to improve them 1. Biological Inspiration ● Understanding the neuron structure ● Synapses and signal transmission ● How biological concepts translate to artificial neurons 2. History of Neural Networks ● Early models (Perceptron) ● Backpropagation and MLPs ● The "AI Winter" and resurgence of neural networks ● Emergence of deep learning 3. Perceptron and Multilayer Perceptrons (MLP) ● Single-layer perceptron limitations ● XOR problem and the need for hidden layers ● MLP architecture 4. Layers and Their Functions ● Input Layer ○ Accepting input data ● Hidden Layers ○ Feature extraction ● Output Layer ○ Producing final predictions 5. Activation Functions


In [88]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

vector_store = FAISS.from_documents(docs, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [89]:
retriever = vector_store.as_retriever(search_type = "mmr", search_kwargs = {"k" : 3})

result = retriever.invoke("What is ANN?")

for r in result:
    print("Page:", r.metadata["page_label"])
    print(r.page_content[:200])
    print("---")

Page: 3
● Stochastic Gradient Descent (SGD) ○ Advantages in large datasets ● Mini-Batch Gradient Descent ○ Balancing between batch and SGD 10. Optimization Algorithms ● Momentum ○ Accelerating SGD ● Nesterov 
---
Page: 16
layers as per the architecture. ○ Output Layer : Generates final predictions, often followed by a softmax function. 4. Types of Transformers A. BERT (Bidirectional Encoder Representations from Transfo
---
Page: 22
● Datasets ○ SQuAD, TriviaQA. C. Machine Translation ● Transformer Models ○ Implementing translation systems without RNNs. ● Datasets ○ WMT datasets. D. Text Summarization ● Abstractive Summarization 
---


In [93]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate

load_dotenv()

prompt = PromptTemplate(
    template="""
    You are a helpful assistant.
    Answer ONLY from the provided context from the PDF.
    If the context is insufficient, say you don't know.
    Mention the page numbers you used.

    Do not add facts that are not written in the context.
    If the context only lists a topic without explaining it, say the document only mentions it.

    {context}

    Question: {question}
    """,
    input_variables=["context", "question"]
)

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)



In [94]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

def format_docs(docs):
    result = "\n\n".join(
        f"[Page {d.metadata["page_label"]}] \n {d.page_content}" for d in docs
    )
    return result


chain = ({
    "context" : retriever | RunnableLambda(format_docs),
    "question" : RunnablePassthrough(),
      }
      | prompt | llm | parser
)

In [95]:
chain.invoke("What is Artificial neural network?")

'The document does not give a direct definition of an artificial neural network. It only lists the term as a heading and then outlines related topics—such as biological inspiration, the history of neural networks, perceptrons, layer functions, and activation functions—without explicitly defining what an artificial neural network is. (Page\u202f1)'

In [96]:
print(repr(docs[5].page_content[:200]))

'7. Loss Functions ● Cross-Entropy Loss for Classification ● Mean Squared Error for Regression 8. CNN Architecture Layer Stacking ● Convolutional -> Activation -> Pooling Feature Maps ● Understanding d'


In [97]:
for q in ["What is ANN?", "What is Artificial Neural Network?"]:
    print(q)
    for r in retriever.invoke(q):
        print("  Page", r.metadata["page_label"])

What is ANN?
  Page 3
  Page 16
  Page 22
What is Artificial Neural Network?
  Page 1
  Page 11
  Page 15


In [98]:
rewrite_prompt = PromptTemplate(
    template="""Rewrite the question as a clear, standalone search query for a document.
Expand any abbreviations into their full form.
Return only the rewritten question.

Question: {question}""",
    input_variables=["question"]
)

rewriter = rewrite_prompt | llm | parser

print(rewriter.invoke("What is ANN?"))

What is an artificial neural network?


In [99]:
chain = ({
    "context": rewriter | retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough(),
} | prompt | llm | parser)

In [100]:
chain.invoke("What is RNN?")

'The document does not provide a definition of “RNN.” It only mentions RNNs in the context of a bidirectional RNN architecture, forward and backward passes, combining outputs, and several applications (language modeling, sentiment analysis, POS tagging, time‑series forecasting) on **page\u202f11**. Since no explicit description of what an RNN is is given, I don’t know.'

In [101]:
chain.invoke("what is CNN?")

'CNN stands for **Convolutional Neural Network**.  According to the document, a CNN is a deep‑learning model that processes data (typically images) through a series of specialized layers:\n\n* **Convolution Operation** – uses kernels/filters to detect edges and extract features from the input (Page\u202f5).  \n* **Activation Functions** – such as ReLU (and its variants) are applied after convolutions to introduce non‑linearity (Page\u202f5).  \n* **Pooling Layers** – perform dimensionality reduction and provide translation invariance via max or average pooling (Page\u202f5).  \n* **Fully Connected Layers** – after the convolutional and pooling stages, the feature maps are flattened and fed into dense layers for final classification or regression (Page\u202f5).  \n\nThe document also lists several well‑known CNN architectures (AlexNet, VGG, Inception/GoogLeNet, ResNet, MobileNets) that illustrate how these components are combined in practice (Page\u202f7).  \n\n**Thus, a CNN is a neural

In [102]:
def ask(question):
    search_query = rewriter.invoke({"question": question})
    found = retriever.invoke(search_query)
    context = format_docs(found)

    answer = (prompt | llm | parser).invoke(
        {"context": context, "question": question}
    )

    pages = sorted({d.metadata["page"] + 1 for d in found})
    return {"answer": answer, "pages": pages, "search_query": search_query}

In [103]:
result = ask("What is RNN?")
print(result["search_query"])
print(result["answer"])
print("Pages retrieved:", result["pages"])

What is a Recurrent Neural Network?
The document does not provide a definition of “RNN.” It only mentions RNNs in the context of a bidirectional RNN architecture (forward and backward passes, combining outputs) and lists some applications such as language modeling, sentiment analysis, POS tagging, and time‑series forecasting (see Page 11). Since no explicit description of what an RNN is is given, I don’t know.
Pages retrieved: [1, 11, 16]


In [104]:
ask("Who won the 2011 cricket world cup?")

{'answer': "I don't know. The provided document does not contain any information about the winner of the 2011 cricket World Cup. (No relevant page found)",
 'pages': [8, 16, 21],
 'search_query': 'Winner of the 2011 Cricket World Cup'}

In [105]:
queries = [
    "What is an artificial neural network?",
    "What activation functions are covered?",
    "Who won the 2011 cricket world cup?",
]

for q in queries:
    print(q)
    for doc, score in vector_store.similarity_search_with_score(q, k=3):
        print("  score:", round(float(score), 2), "| page", doc.metadata["page_label"])

What is an artificial neural network?
  score: 1.01 | page 1
  score: 1.13 | page 2
  score: 1.25 | page 9
What activation functions are covered?
  score: 1.22 | page 15
  score: 1.27 | page 1
  score: 1.27 | page 2
Who won the 2011 cricket world cup?
  score: 1.76 | page 21
  score: 1.78 | page 8
  score: 1.81 | page 7


In [106]:
print(rewriter.invoke({"question": "What is RF?"}))
print(rewriter.invoke({"question": "What is RF in machine learning?"}))

What is radio frequency?
What is Random Forest in machine learning?


In [107]:
rewrite_with_context = PromptTemplate.from_template(
    """Rewrite the question as a clear, standalone search query for a document.
The documents being searched are: {doc_names}
Expand any abbreviations into their full form, choosing the meaning that fits these documents.
Return only the rewritten question.

Question: {question}"""
) | llm | parser

print(rewrite_with_context.invoke({
    "question": "What is RF?",
    "doc_names": "Random forest & XGBoost.pdf"
}))

What is Random Forest?


In [108]:
summ_map = PromptTemplate.from_template(
    "Summarize the key points of this text in 3-4 bullet points:\n\n{text}"
) | llm | parser

summ_reduce = PromptTemplate.from_template(
    "Combine these partial summaries into one clear summary of the whole "
    "document in about 150 words:\n\n{summaries}"
) | llm | parser

def summarize(chunks, group_size=6):
    groups = [
        "\n".join(c.page_content for c in chunks[i:i + group_size])
        for i in range(0, len(chunks), group_size)
    ]
    partials = [summ_map.invoke({"text": g}) for g in groups]
    final = summ_reduce.invoke({"summaries": "\n\n".join(partials)})
    return final, len(groups)

summary, n_groups = summarize(docs)
print("Groups:", n_groups)
print(summary)

Groups: 4
The document surveys deep‑learning fundamentals and their evolution from simple perceptrons to today’s large‑scale transformer models. It begins with artificial neural‑network basics—biological inspiration, MLP architecture, activation functions, forward propagation, loss, back‑propagation, and modern optimizers (Momentum, Adam). Training tricks such as smart weight‑initialization, batch‑norm, gradient‑clipping, dropout, early stopping, and hyper‑parameter tuning are presented. Convolutional networks are introduced to address MLP limitations on images, covering convolutions, kernels, stride/padding, pooling, and preprocessing/augmentation, followed by a timeline of landmark CNNs: LeNet‑5, AlexNet, VGG, Inception, ResNet, and MobileNet. Transfer learning with ImageNet‑pretrained models enables object detection, segmentation, and generative tasks (autoencoders, VAEs, GANs). Recurrent networks and their gated variants (LSTM/GRU) are discussed, leading to seq2seq with attention m

In [110]:
from pydantic import BaseModel, Field
from typing import List

class MCQ(BaseModel):
    question: str
    options: List[str] = Field(description="exactly 4 answer options")
    answer: int = Field(description="position of the correct option, 0 to 3")
    explanation: str = Field(description="one short sentence")

class Quiz(BaseModel):
    questions: List[MCQ]

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.exceptions import OutputParserException

quiz_parser = PydanticOutputParser(pydantic_object=Quiz)

quiz_chain = PromptTemplate.from_template(
    """Write {n} multiple-choice questions based ONLY on the text below.
Each question needs exactly 4 options and exactly one correct option.
Do not use outside knowledge.

{format_instructions}

Text:
{text}""",
    partial_variables={"format_instructions": quiz_parser.get_format_instructions()},
) | llm | quiz_parser

def make_quiz(chunks, n=3, tries=2):
    step = max(1, len(chunks) // 4)
    picked = chunks[::step][:4]
    text = "\n\n".join(c.page_content for c in picked)

    for attempt in range(tries):
        try:
            result = quiz_chain.invoke({"n": n, "text": text})
        except OutputParserException:
            continue
        good = [
            q.model_dump() for q in result.questions
            if len(q.options) == 4 and 0 <= q.answer <= 3
        ]
        if good:
            return good
    return []

quiz = make_quiz(docs)
for q in quiz:
    print(q)

{'question': 'Which loss function is commonly used for classification problems?', 'options': ['Cross-Entropy Loss', 'Mean Squared Error', 'Hinge Loss', 'Kullback-Leibler Divergence'], 'answer': 0, 'explanation': 'Cross-Entropy Loss is the standard loss for classification tasks.'}
{'question': 'What are the main purposes of data augmentation when training CNNs?', 'options': ['To reduce overfitting only', 'To increase dataset diversity only', 'Both to reduce overfitting and increase dataset diversity', 'To decrease the model size'], 'answer': 2, 'explanation': 'Data augmentation both reduces overfitting and expands dataset diversity.'}
{'question': 'In the Transformer decoder stack, which component prevents a position from attending to subsequent positions?', 'options': ['Multi-head self-attention layer', 'Masked multi-head self-attention layer', 'Feedforward network', 'Encoder‑decoder attention layer'], 'answer': 1, 'explanation': 'The masked multi-head self-attention layer blocks atten